# Build Stan input dictionary

Convert `combined_dataframe_pilot.csv` into `stan_fit.pkl` for the Stan models in `stan_file/`.

Each subject-session pair is treated as an independent Stan unit, matching the original `save_dat_to_stanfit_list(1).ipynb` workflow.

In [9]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import utils
import os

## Paths

The notebook works when launched either from the project root or from `python_script/`.

In [10]:
# Preparation
dataset = 'TUS' # TUS, joystick, speakup, africa
split_dataset = False

# Loads the datapath, subject_id's and session numbers based on whether we're analysing the study or pilot data
if dataset == 'TUS' or dataset == 'joystick':
    raw_output_path = '/Volumes/project/3025011.02/raw'
    unique_subject_ids = [39, 48, 49, 52, 53, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
    unique_sessions = [2, 3, 4]
elif dataset == 'pilot':
    raw_output_path = '/Volumes/project/3025011.02/long_experiment_data/phd_1_task/experiment_output'
    unique_subject_ids = [5, 10, 14, 15, 16, 20, 22, 23, 25, 28, 29, 37, 38]
    unique_sessions = [2, 3, 4]
elif dataset=='speakup':
    raw_output_path = '/Volumes/project/3025011.01/4kenneth/190526/raw'
    unique_subject_ids = [2, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 53]
    unique_sessions = [3]
elif dataset=='africa':
    raw_output_path = '/Volumes/project/3025011.01/4kenneth/RL_task-speakup_version_martin_southafrica/experiment_output'
    unique_subject_ids = [2, 3, 4, 5, 6, 8, 9, 10, 12, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
    unique_sessions = [1]

# Make output folder
output_dir = f'/Volumes/project/3025011.02/modeling'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

combined_results_dataframe = utils.import_functions.main(raw_output_path, unique_subject_ids, unique_sessions, dataset)

sub-039 did not complete session-02
sub-039 did not complete session-03
sub-039 did not complete session-04
sub-048 did not complete session-03
sub-048 did not complete session-04
sub-049 did not complete session-02
sub-049 did not complete session-03
sub-049 did not complete session-04
sub-062 did not complete session-02
sub-062 did not complete session-03
sub-062 did not complete session-04
sub-065 did not complete session-04
sub-069 did not complete session-02
sub-069 did not complete session-03
sub-069 did not complete session-04
These participants will be excluded: {'sub-059_session-03'}



In [11]:
OUTPUT_PATH = Path(output_dir) / "stan_fit.pkl"

print(f"Output: {OUTPUT_PATH}")

Output: /Volumes/project/3025011.02/modeling/stan_fit.pkl


## Load and validate trial data

Only columns required by the Stan conversion are checked for missing values. This avoids deleting valid trials because unrelated derived columns are incomplete.

In [12]:
raw_df = combined_results_dataframe

required_columns = [
    "subject_id",
    "session",
    "trial",
    "response",
    "volatility",
    "valence",
    "subjectively_correct",
    "RT_s",
]

missing_columns = sorted(set(required_columns) - set(raw_df.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# The original workflow excludes session 0. The pilot file currently contains sessions 2-4.
select_df = raw_df.loc[raw_df["session"] != 0, required_columns].copy()
select_df = select_df.sort_values(
    ["subject_id", "session", "trial"], kind="stable"
).reset_index(drop=True)

missing_required = select_df[required_columns].isna().sum()
missing_required = missing_required[missing_required > 0]
if not missing_required.empty:
    raise ValueError(
        "Required modeling columns contain missing values:\n"
        + missing_required.to_string()
    )

duplicate_trials = select_df.duplicated(["subject_id", "session", "trial"])
if duplicate_trials.any():
    raise ValueError(
        f"Found {int(duplicate_trials.sum())} duplicate subject-session-trial rows."
    )

print(f"Loaded {len(raw_df):,} rows; retained {len(select_df):,} modeling trials.")

Loaded 19,291 rows; retained 19,291 modeling trials.


## Encode variables

| Stan variable | Source | Encoding |
|---|---|---|
| `choice` | `response` | avoid = 0, approach = 1 |
| `vol` | `volatility` | stable = -0.5, volatile = 0.5 |
| `cue` | `valence` | angry = -0.5, happy = 0.5 |
| `cong` | `response` + `valence` | actual choice incongruent = -0.5, congruent = 0.5 |
| `outcome` | `subjectively_correct` | False = 0, True = 1 |
| `rt` | `RT_s` | response time in seconds |
| `outlier` | `RT_s` | RT < 0.2 s = 1; otherwise 0 |
| `minrt`, `maxrt` | `RT_s` | subject-session extrema among non-outlier RTs |

In [13]:
def map_categories(series, mapping, column_name, dtype):
    normalized = series.astype(str).str.strip().str.lower()
    encoded = normalized.map(mapping)
    if encoded.isna().any():
        unknown = sorted(normalized.loc[encoded.isna()].unique().tolist())
        raise ValueError(f"Unexpected values in {column_name!r}: {unknown}")
    return encoded.to_numpy(dtype=dtype)


choice = map_categories(
    select_df["response"], {"avoid": 0, "approach": 1}, "response", np.int64
)
vol = map_categories(
    select_df["volatility"], {"stable": -0.5, "volatile": 0.5},
    "volatility", np.float64
)
cue = map_categories(
    select_df["valence"], {"angry": -0.5, "happy": 0.5},
    "valence", np.float64
)
if pd.api.types.is_bool_dtype(select_df["subjectively_correct"]):
    outcome = select_df["subjectively_correct"].to_numpy(dtype=np.int64)
else:
    outcome = map_categories(
        select_df["subjectively_correct"], {"false": 0, "true": 1},
        "subjectively_correct", np.int64
    )

rt = select_df["RT_s"].to_numpy(dtype=np.float64)
if not np.isfinite(rt).all() or np.any(rt <= 0):
    raise ValueError("RT_s must contain finite response times greater than zero.")

# DDM fits treat sub-200 ms responses as RT outliers and use their
# choice-only (RT-integrated) likelihood contribution.
outlier = (rt < 0.2).astype(np.int64)

# Match classify_congruence() in save_dat_to_stanfit_list(1).ipynb:
# happy + approach and angry + avoid are choice-congruent.
cong = np.where(
    ((cue == 0.5) & (choice == 1)) | ((cue == -0.5) & (choice == 0)),
    0.5,
    -0.5,
).astype(np.float64)

# Note: the CSV column named 'congruency' describes whether the currently
# most rewarding response is cue-congruent. It is a different construct and
# is intentionally not used for the Stan variable 'cong' here.

## Create subject-session indices and Stan dictionary

In [14]:
unit = (
    select_df["subject_id"].astype(str)
    + "_ses-"
    + select_df["session"].astype(str)
)
codes, unique_units = pd.factorize(unit, sort=False)
sub_list = codes.astype(np.int64) + 1

# The contaminant uniform RT distribution is defined over each
# subject-session's non-outlier RT range.
minrt = np.empty(len(unique_units), dtype=np.float64)
maxrt = np.empty(len(unique_units), dtype=np.float64)
for subject_index in range(1, len(unique_units) + 1):
    valid_rt = rt[(sub_list == subject_index) & (outlier == 0)]
    if valid_rt.size == 0:
        raise ValueError(
            f"Subject-session {subject_index} has no non-outlier RTs for the DDM."
        )
    minrt[subject_index - 1] = valid_rt.min()
    maxrt[subject_index - 1] = valid_rt.max()

save_fit = {
    "ns": int(len(unique_units)),
    "nt": int(len(select_df)),
    "sub_list": sub_list,
    "choice": choice,
    "vol": vol,
    "cue": cue,
    "cong": cong,
    "outcome": outcome,
    "rt": rt,
    "outlier": outlier,
    "minrt": minrt,
    "maxrt": maxrt,
}

# Structural checks shared by the Stan models in stan_file/.
trial_keys = [
    "sub_list", "choice", "vol", "cue", "cong", "outcome", "rt", "outlier"
]
subject_keys = ["minrt", "maxrt"]
assert all(len(save_fit[key]) == save_fit["nt"] for key in trial_keys)
assert save_fit["sub_list"].min() == 1
assert save_fit["sub_list"].max() == save_fit["ns"]
assert set(np.unique(save_fit["choice"])) <= {0, 1}
assert set(np.unique(save_fit["outcome"])) <= {0, 1}
assert set(np.unique(save_fit["outlier"])) <= {0, 1}
assert set(np.unique(save_fit["cue"])) <= {-0.5, 0.5}
assert set(np.unique(save_fit["vol"])) <= {-0.5, 0.5}
assert set(np.unique(save_fit["cong"])) <= {-0.5, 0.5}
assert all(len(save_fit[key]) == save_fit["ns"] for key in subject_keys)
assert np.all(save_fit["minrt"] > 0.1)
assert np.all(save_fit["maxrt"] > save_fit["minrt"])
assert np.all(np.diff(save_fit["sub_list"]) >= 0), (
    "Each subject-session unit must occupy one contiguous trial block."
)

unit_lookup = (
    select_df.assign(sub_list=sub_list, unit=unit)
    .groupby(["sub_list", "unit", "subject_id", "session"], sort=False)
    .agg(
        n_trials=("trial", "size"),
        n_rt_outliers=("RT_s", lambda values: int((values < 0.2).sum())),
        first_trial=("trial", "min"),
        last_trial=("trial", "max"),
    )
    .reset_index()
)

print(f"ns = {save_fit['ns']}")
print(f"nt = {save_fit['nt']:,}")
unit_lookup

ns = 44
nt = 19,291


,sub_list,unit,subject_id,session,n_trials,n_rt_outliers,first_trial,last_trial
0,1,48_ses-2,48,2,436,0,8,451
1,2,52_ses-2,52,2,442,0,8,451
2,3,52_ses-3,52,3,430,0,8,451
3,4,52_ses-4,52,4,439,0,8,451
4,5,53_ses-2,53,2,440,0,8,451
5,6,53_ses-3,53,3,441,0,8,451
6,7,53_ses-4,53,4,434,0,8,451
7,8,54_ses-2,54,2,440,0,8,451
8,9,54_ses-3,54,3,443,0,8,451
9,10,54_ses-4,54,4,444,0,8,451


## Save and reload the pickle

In [15]:
with OUTPUT_PATH.open("wb") as file:
    pickle.dump(save_fit, file, protocol=pickle.HIGHEST_PROTOCOL)

with OUTPUT_PATH.open("rb") as file:
    reloaded_fit = pickle.load(file)

assert reloaded_fit["ns"] == save_fit["ns"]
assert reloaded_fit["nt"] == save_fit["nt"]
assert all(
    np.array_equal(reloaded_fit[key], save_fit[key])
    for key in (*trial_keys, *subject_keys)
)

print(f"Saved and verified: {OUTPUT_PATH}")
print(f"Keys: {list(reloaded_fit.keys())}")

Saved and verified: /Volumes/project/3025011.02/modeling/stan_fit.pkl
Keys: ['ns', 'nt', 'sub_list', 'choice', 'vol', 'cue', 'cong', 'outcome', 'rt', 'outlier', 'minrt', 'maxrt']
